In [ ]:
#install libraries
!pip -q install pandas numpy requests beautifulsoup4 lxml tqdm sentence-transformers faiss-cpu transformers accelerate sentencepiece

In [ ]:
#imports + constants
import re, time, random, json
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://givana.my"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; GivanaRAGBot/1.0)"}

# WooCommerce product category sitemap you used earlier
PRODUCT_CAT_SITEMAP = BASE_URL.rstrip("/") + "/taxonomy-type-product_cat-sitemap-1.xml"

In [ ]:
#Helper: Read URLs from sitemap
def fetch_urls_from_sitemap(sitemap_url, limit=500):
    r = requests.get(sitemap_url, timeout=25, headers=HEADERS)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "xml")
    return [loc.get_text().strip() for loc in soup.find_all("loc")][:limit]

In [ ]:
#Helper: Extract product links from the category listing pages
def extract_product_links_from_listing(listing_url):
    r = requests.get(listing_url, timeout=25, headers=HEADERS)
    if r.status_code != 200:
        return set()
    soup = BeautifulSoup(r.text, "lxml")

    links = set()
    for a in soup.select("a[href*='/product/']"):
        href = a.get("href")
        if not href:
            continue
        full = urljoin(listing_url, href)
        full = full.split("?")[0].split("#")[0].rstrip("/")
        if full.startswith(BASE_URL) and "/product/" in full:
            links.add(full)
    return links

In [ ]:
#Helper: Get product name from product page (H1)
def get_product_name(product_url):
    r = requests.get(product_url, timeout=25, headers=HEADERS)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)
    # fallback
    title = soup.title.get_text(strip=True) if soup.title else ""
    if title:
        return title.split(" - ")[0].strip()
    return None

In [ ]:
def crawl_shop_products(base_url, max_pages=5):
    product_links = set()

    for page in range(1, max_pages + 1):
        if page == 1:
            url = f"{base_url}/shop/"
        else:
            url = f"{base_url}/shop/page/{page}/"

        try:
            r = requests.get(url, timeout=20, headers=HEADERS)
            if r.status_code != 200:
                break

            soup = BeautifulSoup(r.text, "lxml")

            links_on_page = 0
            for a in soup.select("a[href*='/product/']"):
                href = a.get("href")
                if href:
                    full = urljoin(url, href)
                    full = full.split("?")[0].split("#")[0].rstrip("/")
                    if full.startswith(BASE_URL):
                        product_links.add(full)
                        links_on_page += 1

            # if no products found on page, stop pagination
            if links_on_page == 0:
                break

            time.sleep(0.5)

        except Exception as e:
            print("Shop crawl error:", e)
            break

    return sorted(product_links)

In [ ]:
current_product_urls = crawl_shop_products(BASE_URL, max_pages=5)

print("Products found from shop:", len(current_product_urls))
print(current_product_urls[:15])

**The above two cells are extracting the products from the shop with pagination and run shop crawler. In that case, it scrapes the products where the customers actually see and it has less server load and less timeout risk.**

In [ ]:
#Build products_df from the 9 urls
#Extract product names (H1) and create products_df
def get_product_name(product_url):
    r = requests.get(product_url, timeout=25, headers=HEADERS)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)
    title = soup.title.get_text(strip=True) if soup.title else ""
    return title.split(" - ")[0].strip() if title else None

products = []
for u in tqdm(current_product_urls):
    try:
        name = get_product_name(u)
        if name:
            products.append({"name": name, "url": u})
        time.sleep(0.2)
    except Exception as e:
        print("Name fail:", u, e)

products_df = pd.DataFrame(products).drop_duplicates().sort_values("name").reset_index(drop=True)
print("Current products found:", len(products_df))
products_df

In [ ]:
#Crawl only 9 products
#Scraper
def clean_text(t: str) -> str:
    return re.sub(r"\s+", " ", t).strip()

def scrape_page(url: str) -> dict:
    r = requests.get(url, timeout=25, headers=HEADERS)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    for tag in soup(["script","style","noscript","header","footer","nav"]):
        tag.decompose()

    title = soup.title.get_text(strip=True) if soup.title else url
    text = clean_text(soup.get_text(" "))
    return {"url": url, "title": title, "text": text[:15000]}

In [ ]:
#Build docs_raw
docs_raw = []
for u in tqdm(products_df["url"].tolist()):
    try:
        docs_raw.append(scrape_page(u))
        time.sleep(0.25)
    except Exception as e:
        print("Scrape fail:", u, e)

print("docs_raw pages:", len(docs_raw))
print(docs_raw[0]["title"] if docs_raw else "No docs scraped")

In [ ]:
#Chunking
def chunk_text(text: str, chunk_size=220, overlap=40):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+chunk_size]))
        i += max(1, chunk_size - overlap)
    return chunks

chunks = []
for d in docs_raw:
    for idx, ch in enumerate(chunk_text(d["text"], chunk_size=220, overlap=40)):
        chunks.append({
            "chunk_id": f'{d["url"]}__{idx}',
            "url": d["url"],
            "title": d["title"],
            "text": ch
        })

chunks_df = pd.DataFrame(chunks)
print("chunks_df:", chunks_df.shape)
chunks_df.head(3)

In [ ]:
#Embedding and FAISS
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = chunks_df["text"].tolist()
emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
emb = np.array(emb).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

print("FAISS index size:", index.ntotal)

In [ ]:
#Retriever and clean retriever
def retrieve(query: str, k=5):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    out = []
    for score, i in zip(scores[0], ids[0]):
        row = chunks_df.iloc[int(i)]
        out.append({
            "score": float(score),
            "url": row["url"],
            "title": row["title"],
            "text": row["text"]
        })
    return out

NOISE = [
    "add to cart", "reviews", "be the first to review", "your rating",
    "cancel reply", "email address will not be published", "sku", "quantity",
]

def clean_context(text: str) -> str:
    tl = text.lower()
    for p in NOISE:
        tl = tl.replace(p, " ")
    tl = re.sub(r"\s+", " ", tl).strip()
    return tl

def retrieve_clean(query: str, k=3):
    res = retrieve(query, k=k)
    cleaned = []
    for r in res:
        ct = clean_context(r["text"])
        if len(ct) > 120:
            cleaned.append({**r, "clean_text": ct})
    return cleaned[:k]

In [ ]:
#Test retrieval
test_name = products_df["name"].iloc[0]
hits = retrieve_clean(test_name, k=3)
print("Test product:", test_name)
for h in hits:
    print("\n---", round(h["score"], 3), h["url"])
    print(h["clean_text"][:220], "...")

In [ ]:
#Load the LLM
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LLM_NAME = "google/flan-t5-base"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME)

llm_device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(llm_device)

print("Loaded:", LLM_NAME, "| device:", llm_device)

In [ ]:
#Stable generation outputs (No weird outputs)
def llm_generate(prompt: str, max_new_tokens=170):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(llm_device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5,
        repetition_penalty=1.2,
        early_stopping=True
    )
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
#RAG Caption using retrieved context
def rag_caption_llm(product_name, post_type, theme, k=3):
    ctx = retrieve_clean(f"{product_name} {theme} {post_type}", k=k)
    context = "\n".join([f"- {c['clean_text'][:240]}" for c in ctx])[:1100]

    prompt = f"""
You write social media captions for Givana, a Malaysian online gift shop.

Rules:
- English only
- Warm, premium, minimal
- No unrelated stories, no country names, no random names
- 2 short paragraphs maximum
- 1 emoji maximum
- Mention the product name exactly once
- End with ONE soft CTA: "Order on our website" OR "WhatsApp us for customisation"
- Create 8 to 12 hashtags (Malaysia + gifting + occasion). Avoid repeating the exact same hashtag set every time.

Post type: {post_type}
Theme/occasion: {theme}
Product: {product_name}

Use only facts from this context:
{context}

Return exactly in this format:
Caption: ...
Hashtags: ...
"""
    out = llm_generate(prompt)

    caption, hashtags = out, ""
    if "Hashtags:" in out:
        a, b = out.split("Hashtags:", 1)
        caption = a.replace("Caption:", "").strip()
        hashtags = b.strip()
    else:
        hashtags = "#Givana #GivanaMY #GiftMalaysia #GiftIdeas #GiftingMadeEasy #Malaysia #GiftBox #Gifting"

    hashtags = re.sub(r"\s+", " ", hashtags).strip()
    return caption, hashtags, ctx

In [ ]:
#Quick test to make sure if it writes normally
POST_TYPES = ["Product Spotlight","Gifting Tip","FAQ","Relatable Funny","Mini Story","Occasion Reminder"]
THEMES = ["Hari Raya","Birthday","Thank You","Anniversary","Corporate","Just Because"]

p = products_df["name"].iloc[0]
caption, hashtags, ctx = rag_caption_llm(p, "Product Spotlight", "Birthday", k=3)
print(caption)
print("\n", hashtags)

In [ ]:
#Similarity memory
history_embs = []

def max_similarity_to_history(caption: str) -> float:
    if not history_embs:
        return 0.0
    e = embedder.encode([caption], normalize_embeddings=True).astype("float32")[0]
    sims = np.dot(np.vstack(history_embs), e)
    return float(np.max(sims))

def remember_caption(caption: str):
    history_embs.append(embedder.encode([caption], normalize_embeddings=True).astype("float32")[0])

In [ ]:
#Simple time suggestion
def suggest_time(day_idx):
    dow = day_idx % 7
    if dow in [5, 6]:  # weekend
        return random.choice(["11:00", "21:00"])
    return random.choice(["12:30", "20:45"])

In [ ]:
#Generate one unique post with retries
def generate_unique_post(day_idx, product_name, threshold=0.78, tries=20):
    post_type = POST_TYPES[day_idx % len(POST_TYPES)]
    theme = random.choice(THEMES)

    best = None
    best_sim = 1.0

    for _ in range(tries):
        caption, hashtags, ctx = rag_caption_llm(product_name, post_type, theme, k=3)
        sim = max_similarity_to_history(caption)

        if sim < threshold and len(caption) > 50:
            remember_caption(caption)
            return {
                "day": day_idx + 1,
                "product": product_name,
                "post_type": post_type,
                "theme": theme,
                "suggested_time": suggest_time(day_idx),
                "caption": caption,
                "hashtags": hashtags,
                "max_similarity": round(sim, 3),
                "sources": "; ".join(sorted(set([c["url"] for c in ctx])))
            }

        if sim < best_sim:
            best_sim = sim
            best = (caption, hashtags, ctx, post_type, theme)

        post_type = random.choice(POST_TYPES)
        theme = random.choice(THEMES)

    caption, hashtags, ctx, post_type, theme = best
    remember_caption(caption)
    return {
        "day": day_idx + 1,
        "product": product_name,
        "post_type": post_type,
        "theme": theme,
        "suggested_time": suggest_time(day_idx),
        "caption": caption,
        "hashtags": hashtags,
        "max_similarity": round(best_sim, 3),
        "sources": "; ".join(sorted(set([c["url"] for c in ctx])))
    }

In [ ]:
# Cache context per product so we don't retrieve every time
product_context_cache = {}

def get_cached_context(product_name, k=2):
    if product_name not in product_context_cache:
        ctx = retrieve_clean(product_name, k=k)  # simple query works well
        context = "\n".join([f"- {c['clean_text'][:240]}" for c in ctx])[:800]
        product_context_cache[product_name] = (ctx, context)
    return product_context_cache[product_name]

In [ ]:
#faster version of LLM
def rag_caption_llm(product_name, post_type, theme, k=2):
    ctx, context = get_cached_context(product_name, k=k)

    prompt = f"""
You write social media captions for Givana, a Malaysian online gift shop.

Rules:
- English only
- Warm, premium, minimal
- No unrelated stories, no random names or countries
- 2 short paragraphs max
- 1 emoji max
- Mention the product name exactly once
- End with ONE soft CTA: "Order on our website" OR "WhatsApp us for customisation"
- Create 8 to 12 hashtags (vary them)

Post type: {post_type}
Theme/occasion: {theme}
Product: {product_name}

Use only facts from this context:
{context}

Return exactly:
Caption: ...
Hashtags: ...
"""
    out = llm_generate(prompt, max_new_tokens=120)

    caption, hashtags = out, ""
    if "Hashtags:" in out:
        a, b = out.split("Hashtags:", 1)
        caption = a.replace("Caption:", "").strip()
        hashtags = b.strip()
    else:
        hashtags = "#Givana #GivanaMY #GiftMalaysia #GiftIdeas #GiftingMadeEasy #Malaysia #GiftBox #Gifting"

    hashtags = re.sub(r"\s+", " ", hashtags).strip()
    return caption, hashtags, ctx

In [ ]:
#Make generation LLM faster
def llm_generate(prompt: str, max_new_tokens=120):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(llm_device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=2,              # ✅ was 5 (slower)
        repetition_penalty=1.15,
        early_stopping=True
    )
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
def generate_unique_post(day_idx, product_name, threshold=0.80):
    post_type = POST_TYPES[day_idx % len(POST_TYPES)]
    theme = random.choice(THEMES)

    caption, hashtags, ctx = rag_caption_llm(product_name, post_type, theme, k=2)

    sim = max_similarity_to_history(caption)

    remember_caption(caption)

    return {
        "day": day_idx + 1,
        "product": product_name,
        "post_type": post_type,
        "theme": theme,
        "suggested_time": suggest_time(day_idx),
        "caption": caption,
        "hashtags": hashtags,
        "max_similarity": round(sim, 3),
        "sources": "; ".join(sorted(set([c["url"] for c in ctx])))
    }

In [ ]:
history_embs.clear()

products_list = products_df["name"].tolist()

calendar = []
for i in range(30):
    product_name = products_list[i % len(products_list)]
    calendar.append(generate_unique_post(i, product_name, threshold=0.80))

calendar_df = pd.DataFrame(calendar)
calendar_df.head(10)

In [ ]:
#Check similarity stats (quick quality check)
calendar_df["max_similarity"].describe()


In [ ]:
#Export csv and download
calendar_df.to_csv("givana_30_day_calendar_LLM_RAG.csv", index=False)

from google.colab import files
files.download("givana_30_day_calendar_LLM_RAG.csv")